# rulesmith on your own machine: rules for spotting fake job postings

This is the [Colab notebook](quickstart.ipynb) with the judge on your machine instead of an
API. This notebook gives rulesmith 100 labeled job postings and lets it search for a small
program of rules: cheap tests on a posting's fields, and a typed question to a language
model only where the fields cannot settle it. Then it scores what it found on postings it
never saw, beside the plain question asked of every posting.

The judge is Cloudflare's [Clef-flash](https://huggingface.co/Cloudflare/clef-flash), a 9B
model built to answer typed questions, running here: its 4-bit MLX version on a Mac with
Apple silicon and 16 GB or more, or Cloudflare's release on an NVIDIA GPU with about 20 GB.
Its judgments cost nothing. On this notebook's 150 held-out postings it scored 0.720,
against 0.733 for `gpt-4.1`, a difference well within chance (p = 0.88).

The rules are written by `gpt-6.1-sol` through the OpenAI API, so you need a key; about a
hundred calls should come to around a dollar, though we have not measured it. On a smaller
machine, use the Colab notebook.

In [ ]:
# Installs rulesmith with the benchmark datasets, and what runs Clef-flash on this machine,
# unless they are installed already.
import importlib.util
import platform
import subprocess
import sys

APPLE = sys.platform == "darwin" and platform.machine() == "arm64"
packages = []
if importlib.util.find_spec("rulesmith") is None:
    extras = "bench" if APPLE else "bench,clef"
    packages.append(f"rulesmith[{extras}] @ git+https://github.com/bradAGI/rulesmith")
if APPLE and importlib.util.find_spec("mlx_vlm") is None:
    # The versions the MLX port of Clef-flash was tested with.
    packages += ["mlx==0.32.3", "mlx-lm==0.32.0", "mlx-vlm==0.7.4"]
if packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

## Settings

The judge answers the typed questions; the proposer writes the rules. The judge is
Clef-flash on this machine; the proposer is OpenAI's, or any OpenAI-compatible server
given by `PROPOSER_BASE_URL` and `PROPOSER_MODEL`.

In [ ]:
import getpass
import os
from pathlib import Path

# Where the judge listens; any free port.
CLEF_PORT = int(os.environ.get("CLEF_PORT", "8013"))
PROPOSER_URL = os.environ.get("PROPOSER_BASE_URL", "https://api.openai.com/v1")
PROPOSER_MODEL = os.environ.get("PROPOSER_MODEL", "openai/gpt-6.1-sol")
# Extra request fields for the proposer, as JSON; a local Qwen3 needs its thinking turned off.
PROPOSER_OPTIONS = os.environ.get("PROPOSER_OPTIONS", "{}")
# Example scorings each search may spend; 200 is what the published benchmarks used.
BUDGET = int(os.environ.get("SEARCH_BUDGET", "200"))
# Searches to run; the one that scores best on validation is kept.
SEARCHES = int(os.environ.get("SEARCHES", "3"))
SEED = int(os.environ.get("SEED", "0"))

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

import subprocess

# Clef-flash answers one request at a time.
judge = ["--backend", "clef", "--base-url", f"http://127.0.0.1:{CLEF_PORT}", "--max-workers", "1"]


def rulesmith(*args, log=None):
    # Progress goes to a log, not the notebook; an error comes back with what rulesmith said.
    done = subprocess.run(["rulesmith", *map(str, args)], capture_output=True, text=True)
    if log:
        Path(log).write_text(done.stderr)
    if done.returncode:
        raise RuntimeError(f"rulesmith {args[0]} failed:\n{done.stderr[-3000:]}")
print("proposer:", PROPOSER_MODEL, "at", PROPOSER_URL)

## Start the judge

The first run downloads the weights: about 6 GB for the MLX version, about 19 GB for
Cloudflare's release. The server prints its address once the model is loaded, and this
cell waits for that line.

In [ ]:
# The download's progress bars would fill this cell's output.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import snapshot_download


def nvidia_gb():
    if importlib.util.find_spec("torch") is None:
        return 0
    import torch

    return torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0


if APPLE:
    weights = snapshot_download("mlx-community/clef-flash-4bit")
    command = [sys.executable, "-u", f"{weights}/clef_mlx.py", "serve", "--port", str(CLEF_PORT), "--quiet"]
elif nvidia_gb() >= 20:
    command = ["rulesmith", "clef-serve", "--port", str(CLEF_PORT)]
else:
    raise RuntimeError(
        "Clef-flash needs a Mac with Apple silicon, or an NVIDIA GPU with about 20 GB; "
        "on this machine, use the Colab notebook instead."
    )

clef_log = open("clef.log", "w")
clef = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=clef_log, text=True)
# Both servers print one line with their address once the model is loaded.
for line in clef.stdout:
    if "http://" in line:
        print("judge ready:", line.strip())
        break
else:
    raise RuntimeError("Clef-flash stopped before it was ready:\n" + Path("clef.log").read_text()[-3000:])

## The task

Job postings from the [EMSCAD](https://huggingface.co/datasets/victor/real-or-fake-fake-jobposting-prediction)
dataset, sampled half fraudulent and half genuine: 50 to learn from, 50 to choose between
candidate programs, and 150 held out for the end. Each posting has text fields and a few
structured ones, such as whether it shows a company logo.

In [ ]:
import json
from collections import Counter

from rulesmith.bench import DATASETS, sample

task = sample(DATASETS["jobs"], 50, 50, 150, SEED)
Path("task.json").write_text(task.model_dump_json(indent=2, exclude_none=True))

print(task.description)
print("labels:", task.labels)
print("held out:", dict(Counter(example.label for example in task.test)))
print("fields of a posting:", sorted(task.train[0].state))

## The baseline: ask the model about every posting

One rule, which asks the judge the task's question over its labels.

In [ ]:
Path("direct.rules").write_text(f'always: ask "{task.description}" [task labels]\n')
rulesmith("evaluate", "task.json", "direct.rules", "--split", "test", *judge, "--output", "direct.json")

## The search

rulesmith starts from that one rule. It first mines rules a single field settles, then GEPA
asks the proposer for revisions and keeps the ones that score better on the 50 validation
postings. Each judge call costs a little of an example's score, so a rule that settles a
posting without one is worth something, and the starting question is kept unless a
winner's gain passes a sign test. These are the settings the benchmarks used.

One search can come back empty-handed: the proposer answers differently each time, and in
the benchmarks one search in three found no rule worth keeping. So this runs three, and
keeps the one that scored best on the validation postings; the held-out postings play no
part in the choice. Each search's log goes to `search-N.log`.

In [ ]:
scores = {}
for n in range(1, SEARCHES + 1):
    rulesmith(
        "optimize", "task.json", "direct.rules", *judge,
        "--reflection-model", PROPOSER_MODEL, "--reflection-base-url", PROPOSER_URL,
        "--reflection-options", PROPOSER_OPTIONS,
        "--mine", "--call-cost", "0.02", "--significance", "0.1",
        "--max-metric-calls", BUDGET, "--seed", SEED + n, "--output", f"search-{n}",
        log=f"search-{n}.log",
    )
    report = json.loads(Path(f"search-{n}/report.json").read_text())
    scores[n] = report["validation_score"]
    # A search's score is validation accuracy less 0.02 for every model call it made.
    print(f"search {n}: validation score {report['seed_validation_score']:.3f} where it started, "
          f"{report['validation_score']:.3f} for what it kept")

best = max(scores, key=scores.get)
print(f"\nkept search {best}:\n")
print(Path(f"search-{best}/plan.rules").read_text())

Those are the rules search kept. Each `read` names a field, each `when` is a rule tried
in order, and `ask` is the only line that calls the model.

## On postings it never saw

In [ ]:
rulesmith("evaluate", "task.json", f"search-{best}/plan.rules", "--split", "test", *judge, "--output", "searched.json")


def summary(path):
    scorecard = json.loads(Path(path).read_text())
    rows = scorecard["examples"]
    return scorecard["score"], sum(len(row["calls"]) for row in rows) / len(rows)


for name, path in [("ask about every posting", "direct.json"), ("searched rules", "searched.json")]:
    accuracy, calls = summary(path)
    print(f"{name:24}  accuracy {accuracy:.3f}   model calls per posting {calls:.2f}")

In our run on a Mac, all three searches beat where they started, and the rules they kept
scored 0.760 against 0.720 for asking Clef-flash about every posting, while asking it about
fewer than one posting in three.

Rerun it and the numbers move: each search is one draw, and even the best of three can
find no rule worth keeping. The [benchmarks](https://github.com/bradAGI/rulesmith/blob/main/docs/benchmarks.md)
run three seeds per dataset against DSPy's classifiers and a TF-IDF baseline, and say where
rulesmith does not help.

## Next

- `rulesmith run search-N/plan.rules posting.json`, with N the search kept, decides one input, and names the rule that did.
- `rulesmith serve` answers decisions over HTTP; `rulesmith drift` checks production traces
  against what the rules were counted on.
- The [quickstart](https://github.com/bradAGI/rulesmith/blob/main/docs/quickstart.md) covers
  your own task file, and the other judges rulesmith can ask.

In [ ]:
# Stops the judge this notebook started.
clef.terminate()
clef.wait()
print("judge stopped")